# Focal Loss 损失函数详解

## 目录
1. [什么是损失函数](#什么是损失函数)
2. [常见的损失函数](#常见的损失函数)
3. [Focal Loss 介绍](#focal-loss-介绍)
4. [Focal Loss 理论知识](#focal-loss-理论知识)
5. [代码实现](#代码实现)
6. [总结](#总结)

## 什么是损失函数？

### 1. 什么是损失？
在机器学习模型中，对于每一个样本的**预测值与真实值的差**称为损失。

### 2. 什么是损失函数？
是一个用来计算损失的函数。它是一个**非负实值函数**,通常使用 `L(Y, f(x))` 来表示。

### 3. 损失函数的作用
- 度量模型进行每一次预测的好坏（预测值与真实值的差距程度）
- 差距程度越小，则损失越小，该学习模型越好

### 4. 损失函数的使用流程
```
1. 批次训练数据送入模型
2. 前向传播输出预测值
3. 损失函数计算预测值与真实值的差异（损失值）
4. 反向传播更新参数，降低损失
5. 重复迭代，使预测值逼近真实值
```

## 常见的损失函数

### 分类任务损失
- 0-1 loss
- 熵与交叉熵 loss
- softmax loss 及其变种
- KL 散度
- Hinge loss
- Exponential loss
- Logistic loss
- **Focal Loss**

### 回归任务损失
- L1 loss
- L2 loss
- perceptual loss
- 生成对抗网络损失（GAN Loss）
- Wasserstein GAN
- LS-GAN
- Loss-sensitive-GAN

## Focal Loss 介绍

### 引入背景
Focal Loss 的引入主要是为了解决 **one-stage 目标检测中正负样本数量极不平衡**的问题。

### 什么是正负样本不平衡（Class Imbalance）？
在一张图像中：
- **正样本**（能够匹配到目标的候选框）：通常只有 10~50 个
- **负样本**（没有匹配到目标的候选框）：通常有 10,000~100,000 个

大量负样本对训练网络起不到作用，反而会**淹没掉少量但有助于训练的样本**。

### 为什么二阶段检测不需要？
二阶段检测（如 Faster R-CNN）分两步：
1. 第一步生成候选区域（会产生大量负样本）
2. 第二步会选取**特定数量的正负样本**进行检测

因此正负样本不会特别不平衡。

### 传统的解决方法：Hard Negative Mining
不使用所有的负样本训练，而是**选取损失比较大的负样本**来训练。

### Focal Loss 的优势
| 方法 | 效果 |
|------|------|
| Hard Negative Mining | 效果一般 |
| Focal Loss | 效果非常好 |

## Focal Loss 理论知识

### 核心思想
Focal Loss 基于**二分类交叉熵（CE）**，是一个**动态缩放的交叉熵损失**。

通过一个**动态缩放因子**，可以：
- 动态降低训练过程中**易区分样本**的权重
- 将重心快速聚焦在那些**难区分的样本**上

### 理论推导顺序
`Cross Entropy Loss (CE) → Balanced Cross Entropy (BCE) → Focal Loss (FL)`

### 1. Cross Entropy Loss (CE)

二分类交叉熵损失公式：

$$CE(p, y) = \begin{cases} -\log(p) & \text{if } y = 1 \\ -\log(1-p) & \text{otherwise} \end{cases}$$

其中：
- $y \in \{1, -1\}$，分别代表前景和背景
- $p \in [0, 1]$，是模型预测属于前景的概率

### 简化表示
定义：
$$p_t = \begin{cases} p & \text{if } y = 1 \\ 1-p & \text{otherwise} \end{cases}$$

则 CE 可简化为：
$$CE(p, y) = CE(p_t) = -\log(p_t)$$

**注意**：公式中的 log 函数就是 ln 函数（自然对数）

### 2. Balanced Cross Entropy (BCE)

常见的解决类不平衡方法。引入一个**权重因子** $\alpha \in [0, 1]$：
- 当为正样本时，权重因子为 $\alpha$
- 当为负样本时，权重因子为 $1-\alpha$

损失函数改写为：
$$CE(p_t) = -\alpha_t \log(p_t)$$

其中：
$$\alpha_t = \begin{cases} \alpha & \text{if } y = 1 \\ 1-\alpha & \text{otherwise} \end{cases}$$

### 效果
实验表明，当 $\alpha = 0.75$ 时，效果最好。

### 3. Focal Loss (FL)

#### 问题
BCE 虽然解决了正负样本不平衡问题，但**没有区分简单样本和难分样本**。

当易区分负样本超级多时，训练过程会围绕着易区分负样本进行，进而**淹没正样本**。

#### 解决方案：调制因子
引入**调制因子** $(1-p_t)^\gamma$ 来聚焦难分样本：

$$FL(p_t) = -(1-p_t)^\gamma \log(p_t)$$

其中：
- $\gamma \in [0, 5]$，是可调参数
- 当 $\gamma = 0$ 时，退化为 CE 损失函数

#### 调制因子的作用
| 情况 | $p_t$ | 调制因子 | 影响 |
|------|-------|----------|------|
| 易区分样本 | → 1 | → 0 | 损失贡献小 |
| 难区分样本 | → 0 | → 1 | 损失贡献大 |

#### 完整形式
$$FL(p_t) = -\alpha_t (1-p_t)^\gamma \log(p_t)$$

其中：
- $\alpha_t$：调节正负样本损失之间的比例
- $\gamma$：控制简单/难区分样本的权重

### Focal Loss 参数说明

| 参数 | 作用 | 取值范围 |
|------|------|----------|
| $\alpha_t$ | 调节正负样本损失比例 | [0, 1] |
| $\gamma$ | 控制简单/难分样本权重 | [0, 5] |

### 重要特性
1. **调制因子**用于降低易分样本的损失贡献
   - 无论是前景类还是背景类，$p_t$ 越大，样本越容易被区分
   - 调制因子越小，损失贡献越小

2. **$\alpha_t$** 用于调节正负样本损失之间的比例
   - 前景类别使用 $\alpha_t$
   - 背景类别使用 $1-\alpha_t$

3. **$\gamma$ 和 $\alpha_t$** 相互影响
   - 在实际使用中应组合使用
   - 需要根据具体任务调整取值

## 代码实现

In [ ]:
import torch
import torch.nn as nn


class Focal_Loss:
    """
    二分类 Focal Loss
    
    适用于二分类问题，使用 sigmoid 激活函数
    
    Args:
        alpha (float): 正样本权重系数，用于调节正负样本比例。默认 0.25
        gamma (float): 聚焦参数，用于调节难易样本的权重。默认 2.0
    
    Example:
        >>> criterion = Focal_Loss(alpha=0.25, gamma=2)
        >>> preds = model(x)  # sigmoid 后的输出
        >>> loss = criterion(preds, labels)
    """
    def __init__(self, alpha=0.25, gamma=2):
        super(Focal_Loss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, preds, labels):
        """
        计算二分类 Focal Loss
        
        Args:
            preds: sigmoid 的输出结果，范围 [0, 1]，shape: (N, ...)
            labels: 标签，值为 0 或 1，shape: 与 preds 相同
            
        Returns:
            loss: 标量损失值
            
        公式:
            loss = -alpha * (1-p)^gamma * log(p) * y 
                 - (1-alpha) * p^gamma * log(1-p) * (1-y)
        """
        eps = 1e-7
        
        # 正样本损失：-alpha * (1-p)^gamma * log(p)
        loss_1 = -1 * self.alpha * torch.pow((1 - preds), self.gamma) * torch.log(preds + eps) * labels
        
        # 负样本损失：-(1-alpha) * p^gamma * log(1-p)
        loss_0 = -1 * (1 - self.alpha) * torch.pow(preds, self.gamma) * torch.log(1 - preds + eps) * (1 - labels)
        
        loss = loss_0 + loss_1
        return torch.mean(loss)


class Focal_Loss_MultiClass:
    """
    多分类 Focal Loss
    
    适用于多分类问题，使用 softmax 激活函数
    
    Args:
        weight (torch.Tensor): 各类别的权重，shape: (C,)，默认 None
        gamma (float): 聚焦参数，用于调节难易样本的权重。默认 2.0
    
    Example:
        >>> weight = torch.tensor([1.0, 1.0, 1.0])  # 3 个类别
        >>> criterion = Focal_Loss_MultiClass(weight=weight, gamma=2)
        >>> preds = model(x)  # softmax 后的输出
        >>> loss = criterion(preds, labels)
    """
    def __init__(self, weight=None, gamma=2):
        super(Focal_Loss_MultiClass, self).__init__()
        self.gamma = gamma
        self.weight = weight  # 类别权重 tensor
    
    def forward(self, preds, labels):
        """
        计算多分类 Focal Loss
        
        Args:
            preds: softmax 输出结果，shape: (B, C, H, W) 或 (B, C)
            labels: 真实值 (one-hot 编码或与 preds 同 shape)，shape: 与 preds 相同
            
        Returns:
            loss: 标量损失值
            
        公式:
            FL(p_t) = -(1-p_t)^gamma * log(p_t)
        """
        eps = 1e-7
        
        # B*C*H*W -> B*C*(H*W)
        y_pred = preds.view((preds.size()[0], preds.size()[1], -1))
        
        # B*C*H*W -> B*C*(H*W)
        target = labels.view(y_pred.size())
        
        # 计算交叉熵：-log(y_pred) * target
        ce = -1 * torch.log(y_pred + eps) * target
        
        # 计算 focal loss: (1-y_pred)^gamma * ce
        floss = torch.pow((1 - y_pred), self.gamma) * ce
        
        # 乘以类别权重
        if self.weight is not None:
            weight = self.weight.view(1, self.weight.size()[0], -1)
            floss = torch.mul(floss, weight)
        
        # 沿类别维度求和，得到每个位置的损失
        floss = torch.sum(floss, dim=1)
        
        return torch.mean(floss)


# ==================== 使用示例 ====================
if __name__ == '__main__':
    # 示例 1: 二分类 Focal Loss
    print("=" * 50)
    print("=== 二分类 Focal Loss ===")
    print("=" * 50)
    
    N = 8
    # 模拟 sigmoid 输出
    preds = torch.sigmoid(torch.randn(N, 1))
    # 模拟标签
    labels = torch.randint(0, 2, (N, 1)).float()
    
    criterion_binary = Focal_Loss(alpha=0.25, gamma=2)
    loss_binary = criterion_binary.forward(preds, labels)
    
    print(f"preds shape: {preds.shape}, labels shape: {labels.shape}")
    print(f"preds 范围：[{preds.min():.4f}, {preds.max():.4f}]")
    print(f"Focal Loss: {loss_binary.item():.4f}")
    
    # 示例 2: 多分类 Focal Loss
    print("\n" + "=" * 50)
    print("=== 多分类 Focal Loss ===")
    print("=" * 50)
    
    B, C, H, W = 4, 3, 32, 32
    # 模拟 softmax 输出
    preds_mc = torch.softmax(torch.randn(B, C, H, W), dim=1)
    # 模拟 one-hot 标签
    labels_mc = torch.zeros_like(preds_mc)
    labels_mc[:, 0, :, :] = 1  # 假设所有像素都是类别 0
    
    weight = torch.tensor([1.0, 1.0, 1.0])  # 各类别权重
    criterion_mc = Focal_Loss_MultiClass(weight=weight, gamma=2)
    loss_mc = criterion_mc.forward(preds_mc, labels_mc)
    
    print(f"preds shape: {preds_mc.shape}, labels shape: {labels_mc.shape}")
    print(f"Focal Loss: {loss_mc.item():.4f}")
    
    # 示例 3: 不同 gamma 值的对比
    print("\n" + "=" * 50)
    print("=== 不同 gamma 值对比 ===")
    print("=" * 50)
    
    for gamma in [0, 1, 2, 5]:
        criterion_gamma = Focal_Loss(alpha=0.25, gamma=gamma)
        loss_gamma = criterion_gamma.forward(preds, labels)
        print(f"gamma={gamma}: Loss = {loss_gamma.item():.4f}")
    
    # 示例 4: 类别不平衡场景模拟
    print("\n" + "=" * 50)
    print("=== 类别不平衡场景模拟 ===")
    print("=" * 50)
    
    # 创建极度不平衡的样本（90% 负样本，10% 正样本）
    N_imbalance = 100
    preds_imb = torch.sigmoid(torch.randn(N_imbalance, 1))
    labels_imb = torch.zeros(N_imbalance, 1)
    labels_imb[:10] = 1  # 只有 10 个正样本
    
    print(f"正样本数量：{labels_imb.sum().item()}, 负样本数量：{N_imbalance - labels_imb.sum().item()}")
    
    # 比较不同 alpha 值的效果
    for alpha in [0.1, 0.25, 0.5, 0.75]:
        criterion_alpha = Focal_Loss(alpha=alpha, gamma=2)
        loss_alpha = criterion_alpha.forward(preds_imb, labels_imb)
        print(f"alpha={alpha}: Loss = {loss_alpha.item():.4f}")

## 总结

### Focal Loss 核心贡献

| 特性 | 描述 |
|------|------|
| 解决什么问题 | one-stage 目标检测中的正负样本不平衡 |
| 核心思想 | 动态缩放交叉熵，降低易分样本权重 |
| 关键公式 | $FL(p_t) = -\\alpha_t (1-p_t)^\\gamma \\log(p_t)$ |

### 参数选择建议

| 参数 | 推荐值 | 说明 |
|------|--------|------|
| $\\alpha$ | 0.25 | 正样本较少时使用较小值 |
| $\\gamma$ | 2.0 | 控制难易样本的聚焦程度 |

### 代码实现说明

| 类 | 适用场景 | 激活函数 | 输入要求 |
|----|---------|---------|---------|
| `Focal_Loss` | 二分类 | sigmoid | preds: 概率值 [0,1] |
| `Focal_Loss_MultiClass` | 多分类 | softmax | preds: 概率分布 |

### 使用场景
1. **目标检测**（特别是 one-stage 检测器，如 RetinaNet）
2. **类别不平衡**的分类问题
3. **困难样本挖掘**场景
4. **多标签分类**问题
5. **语义分割**任务

### 与其他损失的比较

| 损失函数 | 优点 | 缺点 |
|----------|------|------|
| Cross Entropy | 简单，通用 | 不处理类别不平衡 |
| Weighted CE | 处理类别不平衡 | 不区分难易样本 |
| **Focal Loss** | **同时处理类别不平衡和难易样本** | 需要调节两个参数 |

### 注意事项
1. 二分类版本输入应为 **sigmoid 后的概率值**
2. 多分类版本输入应为 **softmax 后的概率分布**
3. `eps=1e-7` 用于防止 `log(0)` 导致的数值不稳定
4. 类别权重 `weight` 可根据数据集类别频率设置

### 参考文献
- 论文：[Focal Loss for Dense Object Detection](https://arxiv.org/abs/1708.02002)
- 原文作者：Tsung-Yi Lin, Priya Goyal, Ross Girshick, Kaiming He, Piotr Dollár
- 发表会议：ICCV 2017
- GitHub 实现参考：https://github.com/klloss/FocalLoss